# Wedding Planner

This notebook applies advanced concepts from context and state, mcp and multi-agent systems to build a destination wedding planning assistant. The assistant will be a multi-agent system with a coordinating agent and three sub-agents: a travel agent, a venue agent and a DJ (playlist) agent.

- Travel agent: will utilize an external MCP server to find flights to and from a selected destination
- Venue agent: will use the context of the selected destination and Tavily API to search for venues
- DJ: will use context about music preferences and the destination to develop an appropriate playlist for the occasion

#### Design

Orchestrator:
- The orchestrator will be configured with state and memory to capture preferences and decisions about the destination, flight, venue and music. The agent's state will have the following variables:
  - Destination
  - Flight preferences
  - Flight
  - Venue preferences
  - Venue
  - Wedding size
  - Music preferences
- The orchestrator agent will pass these state variables to the sub-agents as context when calling them
- The orchestator will have the following tools
  - Call travel agent
  - Call dj agent
  - Update destination preferences
  - Update venue preferences
  - Update music preferences
  - Update flight preferences
  - Update flight
  - Update venue

Travel Agent:
- The travel agent will have the following tools:
  - Search for flights
  - Evaluate flights
- Travel agent will inherit context from the orchestator agent when calling tools

DJ Agent:
- The DJ agent will have the following tools:
  - Search for music
  - Create playlist
- DJ agent will inherit context from the orchestrator agent when calling tools

## Building the Agentic System

In [1]:
# import modules
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import HumanMessage, ToolMessage

# Web Search Imports
from typing import Dict, Any
from tavily import TavilyClient

from dotenv import load_dotenv


In [2]:
load_dotenv()

True

Creating orchestrator context and state classes to supply relevant information about the wedding

In [3]:
class WeddingPlannerState(AgentState):
    destination: str | None
    wedding_date: str | None
    wedding_size: float | None
    departure_city: str | None
    flight_pref: str | None
    flight: str | None
    venue_pref: str | None
    venue_name: str | None
    music_pref: str | None

### Subagents

#### Using the Departi MCP Server for flights

Utilizing the MultiServerMCPClient class from the langchain_mcp_adapters library to connect

In [4]:
# Creating a kiwi clinet to access the kiwi travel server and get the tools
kiwi_client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

kiwi_tools = await kiwi_client.get_tools()

In [5]:
# Creating the flight subagent
kiwi_agent = create_agent(
    model="claude-haiku-4-5",
    tools=kiwi_tools
)

#append(read_destination)

#### Tavily Web Search For Venues

In [6]:
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Tool to search the web with a specific query"""
    tavily_client = TavilyClient()

    return tavily_client.search(query)

# Creating the tavily agent
tavily_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[web_search]
)

### Orchestrator Agent

Defining the orchestrator tools for performing actions and updating the orchestrator's state

In [7]:
# Read tools
@tool
def read_destination(runtime: ToolRuntime) -> str:
    '''Reads the selected destination from the orchestrator's state'''
    try:
        return runtime.state["destination"]
    except:
        return "No destination could be found in state"

@tool
def read_wedding_size(runtime: ToolRuntime) -> str:
    '''Reads the planned size of the wedding from the orchestrator's state'''
    try:
        return runtime.state["wedding_size"]
    except:
        return "No wedding size could be found in state"

@tool
def read_wedding_date(runtime: ToolRuntime) -> str:
    '''Reads the planned date of the wedding from the orchestrator's state'''
    try:
        return runtime.state["wedding_date"]
    except:
        return "No wedding date could be found in state"

@tool
def read_depature_city(runtime: ToolRuntime) -> str:
    '''Reads the planned departure city of the wedding from the orchestrator's state'''
    try:
        return runtime.state["departure_city"]
    except:
        return "No departure city could be found in state"

@tool
def read_venue_preferences(runtime: ToolRuntime) -> str:
    '''Reads the venue preferences from the orchestrator's state'''
    try:
        return runtime.state["venue_pref"]
    except:
        return "No venue preferences could be found in state"

# Update tools
@tool
def update_destination(destination: str, runtime: ToolRuntime) -> Command:
    """Update the destination city from the user once it has been given"""
    return Command[tuple[()]](update={
        "destination": destination,
        "messages": [ToolMessage("Successfully updated destination",
                                 tool_call_id=runtime.tool_call_id)]
    })

@tool
def update_wedding_date(wedding_date: str, runtime: ToolRuntime) -> Command:
    """Update the wedding date from the user once it has been given"""
    return Command[tuple[()]](update={
        "wedding_date": wedding_date,
        "messages": [ToolMessage("Successfully updated wedding date",
                                 tool_call_id=runtime.tool_call_id)]
    })

@tool
def update_wedding_size(wedding_size: str, runtime: ToolRuntime) -> Command:
    """Update the wedding size from the user once it has been given"""
    return Command[tuple[()]](update={
        "wedding_size": wedding_size,
        "messages": [ToolMessage("Successfully updated wedding size",
                                 tool_call_id=runtime.tool_call_id)]
    })

@tool
def update_departure_city(departure_city: str, runtime: ToolRuntime) -> Command:
    """Update the departure city from the user once it has been given"""
    return Command[tuple[()]](update={
        "departure_city": departure_city,
        "messages": [ToolMessage("Successfully updated departure city",
                                 tool_call_id=runtime.tool_call_id)]
    })

Create the main orchestrator agent to call each of the subagents

In [8]:
@tool
def call_kiwi_agent(destination: str, date: str, departure_city: str) -> str:
    """Tool to call the kiwi agent to find flights"""
    try:
        flights = kiwi_agent.ainvoke({"messages":HumanMessage(content=f"Get flights to {destination} from {departure_city} on {date}")})
        return flights.messages[-1],content
    except:
        "Failed to return flights using the Kiwi MCP server"

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Tool to search the web for venues in the destionation city with the key preferences"""

    try:
        search_results = tavily_agent.invoke({"messages": [HumanMessage(content=query)]})
        return search_results["messages"][-1].content
    except:
        return "Could not perform web search"

In [9]:
orchestrator_prompt = """
You are a wedding planner that helps plan destination weddings.
Use the defined tools to read the context about the wedding and 
to call the Kiwi agent to find flights for the wedding. Ask
follow up questions to the user to get information you need to
call the subagents, and update your state when that information
is received.
"""

wedding_planner = create_agent(
    model="claude-sonnet-5",
    tools=[call_kiwi_agent,
           read_wedding_size,
           read_destination,
           read_wedding_date,
           read_depature_city,
           update_destination,
           update_departure_city,
           update_wedding_date,
           update_wedding_size,
           read_venue_preferences,
           web_search],
    checkpointer=InMemorySaver(),
    state_schema=WeddingPlannerState,
    system_prompt=orchestrator_prompt
)

In [10]:
test = wedding_planner.invoke(
    {'messages':HumanMessage(content="Help me find flights to my wedding")},
    {"configurable": {"thread_id": "1"}},
)

In [11]:
test

{'messages': [HumanMessage(content='Help me find flights to my wedding', additional_kwargs={}, response_metadata={}, id='6b4c74f1-7639-4f87-b450-82a6bc220ccf'),
  AIMessage(content=[{'signature': 'Es8CCpABCBEYAipAHJXctsIp8hkzQwyqlEsUI62sm08hTvMmJGDF0WOJzRXPiRs4UA7lK1ooPADeBKbLjxdnOdrCDuk8/qN1bj4xETIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ3MzZmNjBiNS1hOGI0LTRlNjItOTZhNi03NDViMWVmNWY4NTaoAbaCudQGEgwxrhX9YR85fdNTNI4aDAIBRTpPAl/KxxSA7iIwuhrpA9Jovb3/OEJqyYHu6Q+L3YLprsQVTZrZD1lqaijOv2W/scw/z+xj70vprwisKmyLTZsx775CLLvrKYv6pMMkL7AYFJRsAq/LtovcERBKRt0deTDtbxQ3rdykU6gPCZwYRJD9y9MgbiCG4S3gB1bIgUMQTVf+uyf4QjAenht/G9j8i00KThQkvYSRAIkROPHqyClxTO/i0BvHT7gYAQ==', 'thinking': '', 'type': 'thinking'}, {'id': 'toolu_01Kiydf9Rv69jyhAYEu4LPbD', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_destination', 'type': 'tool_use'}, {'id': 'toolu_01NUXq1rwP9u9W4rEVuXnNsb', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_wedding_date', 'type': 'tool_use'}, {'id': 'toolu_01MqGAr3h4UrP7vjmTYSAQV1',

In [12]:
test = wedding_planner.invoke(
    {'messages':HumanMessage(content="It will be in Paris with 200 people on November 1st 2026. I will fly out of Chicago.")},
    {"configurable": {"thread_id": "1"}},
)

/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/langchain_core/tools/structured.py:97: RuntimeWarning: coroutine 'Pregel.ainvoke' was never awaited
  return self.func(*args, **kwargs)


In [13]:
test

{'messages': [HumanMessage(content='Help me find flights to my wedding', additional_kwargs={}, response_metadata={}, id='6b4c74f1-7639-4f87-b450-82a6bc220ccf'),
  AIMessage(content=[{'signature': 'Es8CCpABCBEYAipAHJXctsIp8hkzQwyqlEsUI62sm08hTvMmJGDF0WOJzRXPiRs4UA7lK1ooPADeBKbLjxdnOdrCDuk8/qN1bj4xETIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ3MzZmNjBiNS1hOGI0LTRlNjItOTZhNi03NDViMWVmNWY4NTaoAbaCudQGEgwxrhX9YR85fdNTNI4aDAIBRTpPAl/KxxSA7iIwuhrpA9Jovb3/OEJqyYHu6Q+L3YLprsQVTZrZD1lqaijOv2W/scw/z+xj70vprwisKmyLTZsx775CLLvrKYv6pMMkL7AYFJRsAq/LtovcERBKRt0deTDtbxQ3rdykU6gPCZwYRJD9y9MgbiCG4S3gB1bIgUMQTVf+uyf4QjAenht/G9j8i00KThQkvYSRAIkROPHqyClxTO/i0BvHT7gYAQ==', 'thinking': '', 'type': 'thinking'}, {'id': 'toolu_01Kiydf9Rv69jyhAYEu4LPbD', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_destination', 'type': 'tool_use'}, {'id': 'toolu_01NUXq1rwP9u9W4rEVuXnNsb', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_wedding_date', 'type': 'tool_use'}, {'id': 'toolu_01MqGAr3h4UrP7vjmTYSAQV1',

In [ ]:
test = wedding_planner.invoke(
    {'messages': HumanMessage(content="Help me find an outdoor venue.")},
    {"configurable": {"thread_id": "1"}},
)

In [15]:
test

{'messages': [HumanMessage(content='Help me find flights to my wedding', additional_kwargs={}, response_metadata={}, id='6b4c74f1-7639-4f87-b450-82a6bc220ccf'),
  AIMessage(content=[{'signature': 'Es8CCpABCBEYAipAHJXctsIp8hkzQwyqlEsUI62sm08hTvMmJGDF0WOJzRXPiRs4UA7lK1ooPADeBKbLjxdnOdrCDuk8/qN1bj4xETIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ3MzZmNjBiNS1hOGI0LTRlNjItOTZhNi03NDViMWVmNWY4NTaoAbaCudQGEgwxrhX9YR85fdNTNI4aDAIBRTpPAl/KxxSA7iIwuhrpA9Jovb3/OEJqyYHu6Q+L3YLprsQVTZrZD1lqaijOv2W/scw/z+xj70vprwisKmyLTZsx775CLLvrKYv6pMMkL7AYFJRsAq/LtovcERBKRt0deTDtbxQ3rdykU6gPCZwYRJD9y9MgbiCG4S3gB1bIgUMQTVf+uyf4QjAenht/G9j8i00KThQkvYSRAIkROPHqyClxTO/i0BvHT7gYAQ==', 'thinking': '', 'type': 'thinking'}, {'id': 'toolu_01Kiydf9Rv69jyhAYEu4LPbD', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_destination', 'type': 'tool_use'}, {'id': 'toolu_01NUXq1rwP9u9W4rEVuXnNsb', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_wedding_date', 'type': 'tool_use'}, {'id': 'toolu_01MqGAr3h4UrP7vjmTYSAQV1',